# Urdu Diffusion-BERT pretraining on Kaggle (→ Hugging Face Hub)

Pretrain a small text-diffusion BERT (MDLM) on Urdu and auto-upload every checkpoint (per-step + final) to a **private Hugging Face model repo**. Eval is out of scope for this notebook — add it back later by running `evaluate.py urdu_sentiment / urdu_nli` against the same HF checkpoint.

Everything (data access + checkpoint storage) uses **one HF token**.

---

## One-time setup (BEFORE opening Kaggle)

1. **HF token**: https://huggingface.co/settings/tokens → **New token** → role **Write** (needed to create / push to a model repo). Copy it.
2. **(Optional, only if you want IndicCorp v2)**: visit https://huggingface.co/datasets/ai4bharat/IndicCorpV2 logged in, click **Agree and access repository** once.

## Kaggle notebook settings

1. Settings → Accelerator → **GPU T4 x2** (or P100).
2. Settings → Internet → **On**.

Run cells 1 → 6 in order.

## Cell 1 — Clone the repo

In [ ]:
!git clone https://github.com/sushanedulloo/Diffusion-Bert.git /kaggle/working/bert
%cd /kaggle/working/bert
!git log -1 --oneline

## Cell 2 — Install dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -U transformers sentencepiece huggingface_hub

## Cell 3 — Hugging Face login + config (EDIT ME)

Paste your **write-scope** HF token. Pick any `repo_id` — it will be auto-created (private) on first checkpoint upload.

The token cached by `login()` is reused for IndicCorp / mmBERT downloads; the trainer's HF callback uses the explicit `--hf_token` you'll pass in Cell 4.

In [ ]:
from huggingface_hub import login, whoami

# === EDIT THESE TWO ===
HF_TOKEN   = "hf_xxx_paste_your_write_token_here"
HF_REPO_ID = "yourname/ur-mdlm-mini"        # auto-created if missing
# ======================

login(token=HF_TOKEN, add_to_git_credential=False)
print("Logged in as:", whoami()["name"])

import os
OUT_DIR = "/kaggle/working/ur_mdlm_mini"
os.environ["HF_TOKEN"] = HF_TOKEN   # so child processes inherit it

## Cell 4 — Pretrain (auto-uploads each checkpoint to HF)

With `--hf_repo_id` set, every saved checkpoint (per-step + final) is:
1. Uploaded to `<HF_REPO_ID>` as `checkpoint_<tag>.pt` (each step is a separate file — full history preserved).
2. **Also** kept locally under `<OUT_DIR>/checkpoint_<tag>/checkpoint.pt`, in case you want to use it from this same session.

**OOM?** Use `--batch_size 16 --gradient_accumulation_steps 8`.
**Time to spare?** Raise `--max_docs 100000` → `500000`, or switch `--model_size mini` → `small`.
**No IndicCorp access?** Add `--no_indiccorp` (Urdu Wikipedia is used alone).

In [ ]:
!python train.py \
    --language ur \
    --training_objective mdlm --noise_schedule cosine \
    --model_size mini \
    --tokenizer_name jhu-clsp/mmBERT-base \
    --max_docs 100000 \
    --num_epochs 1 \
    --batch_size 32 --gradient_accumulation_steps 4 \
    --phase1_max_seq_len 128 --phase2_max_seq_len 128 \
    --learning_rate 2e-4 \
    --dataloader_num_workers 2 \
    --output_dir {OUT_DIR} \
    --hf_repo_id {HF_REPO_ID} \
    --hf_private \
    --hf_token {HF_TOKEN}

## Cell 5 — Verify checkpoints are on HF + locally

Sanity check after Cell 4 finishes: lists local checkpoint dirs and the files in your HF repo.

In [ ]:
import os, glob
from huggingface_hub import HfApi

print('--- Local ---')
for d in sorted(glob.glob(os.path.join(OUT_DIR, 'checkpoint_*'))):
    pt = os.path.join(d, 'checkpoint.pt')
    mb = os.path.getsize(pt) / 1e6 if os.path.exists(pt) else 0
    print(f'  {d}   ({mb:.1f} MB)')

print(f"\n--- HF Hub: {HF_REPO_ID} ---")
for f in HfApi(token=HF_TOKEN).list_repo_files(HF_REPO_ID):
    print(f'  {f}')
print(f"\nBrowse at: https://huggingface.co/{HF_REPO_ID}/tree/main")

## Cell 6 (next session) — Resume from HF Hub

Fresh Kaggle session, or want to continue training from where you stopped? Run Cells 1, 2, 3, then this cell to pull a previously-uploaded checkpoint back from your HF repo. Then re-run Cell 4 with the extra flag:

```
--resume_from_checkpoint {RESUME_CKPT}
```

Defaults to `checkpoint_final.pt`. Change `which` to e.g. `'checkpoint_step_0001000'` to resume from an intermediate step instead.

In [ ]:
import os
from huggingface_hub import hf_hub_download

which = 'checkpoint_final'   # or 'checkpoint_step_0001000', etc.

dst_dir = os.path.join(OUT_DIR, which)
os.makedirs(dst_dir, exist_ok=True)

local = hf_hub_download(
    repo_id   = HF_REPO_ID,
    filename  = f'{which}.pt',
    token     = HF_TOKEN,
    local_dir = dst_dir,
)
# hf_uploader names the file <which>.pt on HF; train.py expects the inner name checkpoint.pt
target = os.path.join(dst_dir, 'checkpoint.pt')
if local != target:
    os.replace(local, target)

RESUME_CKPT = target
print('Downloaded:', target)
print('Pass to train.py:  --resume_from_checkpoint', RESUME_CKPT)